In [1]:
import pandas as pd
import numpy as np

# DATA

In [3]:
df = pd.read_csv("../DATA_intermediate/23_filter_EPN.csv", sep=",", header=0, index_col=0)
print(df.shape)
df.head()

(1467, 29977)


,55749196-29-G/A,55750931-10-C/T,23141276-42-T/A,55764801-6-G/A,100283660-33-C/A,55760984-24-A/G,55749171-62-C/A,55774608-18-A/G,23143410-28-G/A,23145710-35-C/T,...,55842229-60-A/G,23149114-12-G/C,55855639-63-T/C,13501202-27-G/A,55836668-31-A/C,23157730-22-T/G,55846148-65-T/C,55836449-6-T/C,55863991-48-C/T,55816615-37-A/G
E353-B1_3_6_333_001,0.0,0.0,0.0,0.0,NaN,0.0,1.0,0.0,0.0,1.0,...,2.0,1.0,1.0,0.0,2.0,0.0,0.0,NaN,0.0,2.0
E353-B1_3_6_6914_073,0.0,0.0,0.0,NaN,0.0,0.0,0.0,0.0,0.0,2.0,...,2.0,0.0,NaN,1.0,2.0,0.0,0.0,2.0,1.0,2.0
E353-B1_3_16_6917_081,0.0,0.0,0.0,0.0,NaN,0.0,0.0,0.0,0.0,NaN,...,NaN,0.0,0.0,2.0,NaN,1.0,0.0,2.0,NaN,2.0
E353-B3_3_7_6802_478,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,2.0,...,2.0,1.0,0.0,1.0,2.0,1.0,NaN,NaN,0.0,2.0
E353-B3_3_14_6986_371,0.0,0.0,1.0,0.0,NaN,1.0,2.0,1.0,0.0,2.0,...,2.0,1.0,NaN,NaN,0.0,0.0,0.0,2.0,0.0,0.0


# Metadata

Selecting samples from the TRAIN dataset

In [4]:
dmeta = pd.read_csv("../44_train_and_test/00_combine_samples.tsv", sep="\t", header=0)

# Subset only TRAIN
dmeta = dmeta[dmeta['SET'] == 'TRAIN'].reset_index(drop=True)
#dmeta = dmeta.set_index("genotype")
print(dmeta.shape)
dmeta.head()

(18282, 9)


,genotype,POP_ID,SITE_ID,call_rate,good_genotype,value,Trait_name,SET,POP_SITE
0,E60-A_4_1_62_260,332,AC,0.870518,True,559.411198,Average_Ring_Density,TRAIN,332_AC
1,E60-A_4_2_62_261,332,AC,0.852943,True,729.561198,Average_Ring_Density,TRAIN,332_AC
2,E60-A_4_7_62_264,332,AC,0.927686,True,565.111198,Average_Ring_Density,TRAIN,332_AC
3,E60-A_5_4_62_562,332,AC,0.898479,True,623.921198,Average_Ring_Density,TRAIN,332_AC
4,E60-A_5_5_62_563,332,AC,0.886400,True,NaN,Average_Ring_Density,TRAIN,332_AC


In [5]:
# Check if there are any pop-sites with less than 5 samples

#dmeta.groupby(["POP_SITE","Trait_name"])[["POP_ID"]].count().hist()
stats = dmeta.groupby(["POP_SITE","Trait_name"])[["genotype"]].count().reset_index()
stats[stats["genotype"] <= 4]

,POP_SITE,Trait_name,genotype


# Loop for each trait

In [6]:
traits = dmeta['Trait_name'].drop_duplicates().values.tolist()
traits

['Average_Ring_Density',
 'Biomass_Increment',
 'Biomass_Increment_1980',
 'Biomass_Increment_1985',
 'Biomass_Increment_1990',
 'Biomass_Increment_1995',
 'Biomass_Increment_2000',
 'Biomass_Increment_2005',
 'Biomass_Increment_2010',
 'Biomass_Increment_2015',
 'DBH',
 'Height',
 'Rc',
 'Rl',
 'Rr',
 'Rs']

In [12]:
for trait in traits:

    print(trait)
    # Data subset
    dmeta_sub = dmeta[dmeta['Trait_name'] == trait].reset_index(drop=True)
    dmeta_sub = dmeta_sub.set_index("genotype")
    
    # Subset genotypes present in metadata (TRAIN)
    df_sub = df.loc[dmeta_sub.index.values]
    print(df_sub.shape)

    # Calculating frequencies
    df_all = df_sub.groupby(dmeta_sub['POP_SITE']).count()*2
    
    df_alleles = df_sub.groupby(dmeta_sub['POP_SITE']).sum()
    
    df_af = df_alleles.div(df_all)
    print(df_af.shape)

    # Removing SNPs with nan
    cols_with_nan = df_af.columns[df_af.isna().any()]
    df_af = df_af.drop(columns = cols_with_nan)
    print(df_af.shape)

    # Checking snps with no variation
    col_sums = df_af.sum()
    cols_to_exclude = col_sums[col_sums==0].index.values.tolist()
    df_af = df_af.drop(columns = cols_to_exclude)
    print(df_af.shape)
    
    # Saving
    df_af.to_csv('01_frequencies_POP_' + trait + '.tsv', sep='\t', index=True, header=True)

Average_Ring_Density
(1266, 29977)
(115, 29977)
(115, 29464)
(115, 29464)
Biomass_Increment
(1266, 29977)
(115, 29977)
(115, 29464)
(115, 29464)
Biomass_Increment_1980
(683, 29977)
(87, 29977)
(87, 28877)
(87, 28875)
Biomass_Increment_1985
(1114, 29977)
(110, 29977)
(110, 29295)
(110, 29295)
Biomass_Increment_1990
(1250, 29977)
(114, 29977)
(114, 29402)
(114, 29402)
Biomass_Increment_1995
(1266, 29977)
(115, 29977)
(115, 29464)
(115, 29464)
Biomass_Increment_2000
(1266, 29977)
(115, 29977)
(115, 29464)
(115, 29464)
Biomass_Increment_2005
(1259, 29977)
(114, 29977)
(114, 29484)
(114, 29484)
Biomass_Increment_2010
(1238, 29977)
(113, 29977)
(113, 29408)
(113, 29408)
Biomass_Increment_2015
(694, 29977)
(83, 29977)
(83, 29054)
(83, 29053)
DBH
(1257, 29977)
(115, 29977)
(115, 29464)
(115, 29464)
Height
(1257, 29977)
(115, 29977)
(115, 29464)
(115, 29464)
Rc
(1267, 29977)
(115, 29977)
(115, 29465)
(115, 29465)
Rl
(1267, 29977)
(115, 29977)
(115, 29465)
(115, 29465)
Rr
(665, 29977)
(82, 29977